# Step 4. Make binned data for processing by levels, also adjust amplitudes after data stitching for different distances.

In [ ]:
import numpy as np
import cupy as cp
import h5py
from holotomocupy.shift import Shift
from holotomocupy.utils import *

## Init

In [ ]:
ntheta = 1800
show = True
rotation_center_shift = -27.00227
ids = np.arange(0, 1800, 1800 / ntheta).astype('int')

path     = '/data2/vnikitin/atomium/20240924/AtomiumS2/'
pfile    = 'AtomiumS2_HT_007nm'
path_out = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/'
file_out = f'data.h5'


with h5py.File(f'{path_out}/{file_out}') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focusToDetectorDistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    energy = fid['/exchange/energy'][0] 
    shifts = fid['/exchange/shifts'][ids]
    attrs = fid['/exchange/attrs'][ids]
    shape = np.array(fid[f'/exchange/data0'].shape)
    shape_ref = fid['/exchange/data_white_start0'].shape
    shape_dark = fid['/exchange/data_dark0'].shape    

z2 = focusToDetectorDistance - z1
magnifications = focusToDetectorDistance / z1
norm_magnifications = magnifications / magnifications[0]
distances = (z1 * z2) / focusToDetectorDistance * norm_magnifications**2
voxelsize = detector_pixelsize / magnifications[0]

n = shape[1] 
ndist=len(z1)
npsi = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 

from holotomocupy.reader import load_shrink_from_mats
shrink_nd_full = load_shrink_from_mats(path, pfile, ndist, shape[0])
shrink_nd      = shrink_nd_full[ids]

print(f'{energy=}')
print(f'{z1=}')
print(f'{focusToDetectorDistance=}')
print(f'{detector_pixelsize=}')
print(f'{magnifications=}')
print(f'{voxelsize=}')
print(f'{distances=}')
print(f'shrink[0]={shrink_nd[0].tolist()}')

### Read data and write binned ref

In [ ]:
nlevels = 4  # number of bin levels

with h5py.File(f'{path_out}/data.h5', 'a') as fid:
    ref = fid['/exchange/pref'][:ndist].astype('float32')
    r   = fid['/exchange/cshifts_final'][:].astype('float32')
    r[..., 1] += rotation_center_shift

    ref0 = ref.copy()
    for bin in range(nlevels):
        if f'/exchange/pref_{bin}' in fid:
            del fid[f'/exchange/pref_{bin}']
        fid.create_dataset(f'/exchange/pref_{bin}', data=ref0)
        ref0 = 0.5 * (ref0[..., ::2] + ref0[..., 1::2])
        ref0 = 0.5 * (ref0[..., ::2, :] + ref0[..., 1::2, :])

    for bin in range(nlevels):
        for k in range(ndist):
            if f'/exchange/pdata{k}_{bin}' in fid:
                del fid[f'/exchange/pdata{k}_{bin}']

In [ ]:
print(np.amax(np.abs(r)))

### Alignment and intensity correction, save to h5

In [ ]:
cl_shift = Shift(n, npsi, n, npsi, 'complex64')
cref     = cp.array(ref)
r_gpu    = cp.array(r)

v = cp.linspace(0, 1, npad, endpoint=False)
v = v**5 * (126 - 420*v + 540*v**2 - 315*v**3 + 70*v**4)

with h5py.File(f'{path_out}/data.h5', 'a') as fid:
    data_out = [[fid.create_dataset(f'/exchange/pdata{k}_{bin}',
                                    shape=(ntheta, n // 2**bin, n // 2**bin),
                                    dtype='float32')
                 for k in range(ndist)]
                for bin in range(nlevels)]

    srdata = cp.zeros([ndist, npsi, npsi], dtype='float32')

    for j in range(ntheta):
        data = cp.empty([ndist, n, n], dtype='float32')
        for k in range(ndist):
            data[k] = cp.array(fid[f'/exchange/pdata{k}'][j].astype('float32'))

        rdata = data / (cref + 1e-5)

        for k in range(ndist - 1, -1, -1):
            shrink_jk  = float(shrink_nd[j, k])
            eff_mag_jk = float(norm_magnifications[k]) / (1 + shrink_jk)
            mag = cp.array(1.0 / eff_mag_jk, dtype='float32')
            tmp = rdata[k].astype('complex64')
            tmp = cl_shift.curlySback(
                cp.log(tmp[None]).astype('complex64'),
                r_gpu[j:j+1, k], mag
            )[0].real
            tmp /= eff_mag_jk**2
            tmp = cp.exp(tmp)

            padx0 = int((npsi - n / eff_mag_jk) / 2) - int(r[j, k, 1])
            pady0 = int((npsi - n / eff_mag_jk) / 2) - int(r[j, k, 0])
            padx1 = int((npsi - n / eff_mag_jk) / 2) + int(r[j, k, 1])
            pady1 = int((npsi - n / eff_mag_jk) / 2) + int(r[j, k, 0])
            padx0 = min(npsi, max(0, padx0)) + 5
            pady0 = min(npsi, max(0, pady0)) + 5
            padx1 = min(npsi, max(0, padx1)) + 5
            pady1 = min(npsi, max(0, pady1)) + 5

            tmp = cp.pad(tmp[pady0:-pady1], ((pady0, pady1), (0, 0)), 'edge')
            tmp = cp.pad(tmp[:, padx0:-padx1], ((0, 0), (padx0, padx1)),
                         'linear_ramp', end_values=((1, 1), (1, 1)))

            if k < ndist - 1:
                mmm = float(srdata[k + 1][pady0:-pady1, padx0:-padx1].mean() /
                            tmp[pady0:-pady1, padx0:-padx1].mean())
                tmp     *= mmm
                data[k] *= mmm
                wx = cp.ones(npsi, dtype='float32')
                wy = cp.ones(npsi, dtype='float32')
                wx[:padx0]               = 0
                wx[padx0:padx0 + npad]   = v
                wx[-padx1 - npad:-padx1] = 1 - v
                wx[-padx1:]              = 0
                wy[:pady0]               = 0
                wy[pady0:pady0 + npad]   = v
                wy[-pady1 - npad:-pady1] = 1 - v
                wy[-pady1:]              = 0
                w   = cp.outer(wy, wx)
                tmp = tmp * w + srdata[k + 1] * (1 - w)
            srdata[k] = tmp

        if j % 100 == 0:
            print(f'proj {j}/{ntheta}')
            mshow_complex(srdata[0] + 1j * srdata[ndist - 1], show)

        for k in range(ndist):
            datak = data[k]
            for bin in range(nlevels):
                data_out[bin][k][j] = datak.get()
                datak = 0.5 * (datak[::2, :] + datak[1::2, :])
                datak = 0.5 * (datak[:, ::2]  + datak[:, 1::2])